In [ ]:
import logging
import math

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
import torch.nn as nn
from pytorch_lightning import LightningDataModule, Trainer
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader, Dataset

from caveat.label_encoding import TokenAttributeEncoder
from caveat.models.continuous.cvae_lstm import LabelEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

Device: cuda


In [2]:
ys = pd.read_csv(
    "../logs/TRB/cvae/cvae_nrun0/version_5/test_inference/input_attributes.csv"
)

label_encoder = TokenAttributeEncoder(
    config={
        "gender": "nominal",
        "age_group": "nominal",
        "car_access": "nominal",
        "work_status": "nominal",
        "income": "nominal",
    }
)
ys, _ = label_encoder.encode(ys)

zs = pd.read_csv(
    "../logs/TRB/cvae/cvae_nrun0/version_5/test_inference/zs.csv", header=None
).values

Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            


In [11]:
class YZDataset(Dataset):
    def __init__(self, labels: torch.Tensor, zs: torch.Tensor):
        self.zs = zs
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.labels[idx], self.zs[idx]


class DataModule(LightningDataModule):
    def __init__(
        self,
        ys: torch.Tensor,
        zs: torch.Tensor,
        val_split: float = 0.1,
        test_split: float = 0.1,
        batch_size: int = 1024,
        num_workers: int = 1,
        pin_memory: bool = False,
        **kwargs,
    ):
        super().__init__()

        if isinstance(ys, pd.DataFrame):
            ys = ys.values
        if isinstance(zs, pd.DataFrame):
            zs = zs.values
        if isinstance(ys, np.ndarray):
            ys = torch.from_numpy(ys)
        if isinstance(zs, np.ndarray):
            zs = torch.from_numpy(zs).float()
        self.data = YZDataset(ys.long(), zs)

        self.val_split = val_split
        self.test_split = test_split
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.pin_memory = pin_memory
        self.mapping = None

    def setup(self, stage) -> None:
        (self.train_dataset, self.val_dataset, self.test_dataset) = (
            torch.utils.data.random_split(
                self.data,
                [
                    1 - self.val_split - self.test_split,
                    self.val_split,
                    self.test_split,
                ],
            )
        )
        if self.val_split == 0:
            self.val_dataset = self.train_dataset
        if self.test_split == 0:
            self.test_dataset = self.val_dataset

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=True,
            pin_memory=self.pin_memory,
            persistent_workers=True,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=False,
            pin_memory=self.pin_memory,
            persistent_workers=True,
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=False,
            pin_memory=self.pin_memory,
            persistent_workers=True,
        )

In [12]:
def batch(ys: np.ndarray, zs: np.ndarray, batch_size=128, shuffle=True):
    assert len(ys) == len(
        zs
    ), "Input and target data must contain same number of elements"
    n = len(ys)
    if shuffle:
        rand_perm = torch.randperm(n)
        ys = ys[rand_perm]
        zs = zs[rand_perm]

    batches = []
    for i in range(n // batch_size):
        y = ys[i * batch_size : (i + 1) * batch_size]
        z = zs[i * batch_size : (i + 1) * batch_size]

        batches.append((y, z))
    return batches

In [ ]:
class EMALoss(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, running_ema):
        ctx.save_for_backward(input, running_ema)
        input_log_sum_exp = input.exp().mean().log()

        return input_log_sum_exp

    @staticmethod
    def backward(ctx, grad_output):
        input, running_mean = ctx.saved_tensors
        grad = (
            grad_output
            * input.exp().detach()
            / (running_mean + 1e-6)
            / input.shape[0]
        )
        return grad, None


def ema(mu, alpha, past_ema):
    return alpha * mu + (1.0 - alpha) * past_ema


def ema_loss(x, running_mean, alpha):
    t_exp = torch.exp(torch.logsumexp(x, 0) - math.log(x.shape[0])).detach()
    if running_mean == 0:
        running_mean = t_exp
    else:
        running_mean = ema(t_exp, alpha, running_mean.item())
    t_log = EMALoss.apply(x, running_mean)

    return t_log, running_mean

In [14]:
class Mine(nn.Module):
    def __init__(self, net, alpha=0.01):
        super().__init__()
        self.running_mean = 0
        self.alpha = alpha
        self.net = net

    def forward(self, y, z):
        z_marg = z[torch.randperm(y.shape[0])]

        t = self.net(y, z).mean()
        t_marg = self.net(y, z_marg)

        second_term, self.running_mean = ema_loss(
            t_marg, self.running_mean, self.alpha
        )

        return -t + second_term

    def mi(self, y, z):
        with torch.no_grad():
            mi = -self.forward(y, z)
        return mi

    def optimize(self, ys, zs, iters, batch_size, opt=None):

        if opt is None:
            opt = torch.optim.Adam(self.parameters(), lr=1e-4)

        for iter in range(1, iters + 1):
            mu_mi = 0
            for x, y in batch(ys, zs, batch_size):
                opt.zero_grad()
                loss = self.forward(x, y)
                loss.backward()
                opt.step()

                mu_mi -= loss.item()
            # if iter % (iters // 10) == 0:
            #     print(f"It {iter} - MI: {mu_mi / batch_size}")

        final_mi = self.mi(ys, zs)
        return final_mi

In [31]:
class MutualInformationEstimator(pl.LightningModule):
    def __init__(self, net: nn.Module, **kwargs):
        super().__init__()
        self.net = net
        self.energy_loss = Mine(self.net, alpha=kwargs.get("alpha", 0.01))
        self.lr = kwargs.get("lr", 1e-4)

    def forward(self, y, z):
        return self.energy_loss(y, z)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

    def training_step(self, batch, batch_idx):
        y, z = batch
        loss = self.energy_loss(y, z)
        mi = -loss
        self.log_dict({"loss": loss, "mi": mi}, prog_bar=True, logger=True)
        return {"loss": loss, "mi": mi}

    def validation_step(self, batch, batch_idx):
        y, z = batch
        loss = self.energy_loss(y, z)
        mi = -loss
        self.log_dict(
            {"val_loss": loss, "val_mi": mi}, prog_bar=True, logger=True
        )
        return {"val_loss": loss, "val_mi": mi}

    def test_step(self, batch, batch_idx):
        y, z = batch
        loss = self.energy_loss(y, z)
        self.log_dict(
            {"test_loss": loss, "test_mi": -loss}, prog_bar=True, logger=True
        )
        return {"test_loss": loss, "test_mi": -loss}

In [32]:
class MinerNet(nn.Module):
    def __init__(
        self,
        encoder_kwargs,
        hidden_size=128,
        latent_dim=6,
        block_depth=2,
        dropout=0.2,
    ):
        super(MinerNet, self).__init__()

        self.label_embed = LabelEncoder(
            label_embed_sizes=encoder_kwargs["label_embed_sizes"],
            hidden_size=hidden_size,
        )
        self.z_embed = nn.Sequential(
            nn.Linear(in_features=latent_dim, out_features=hidden_size),
            nn.LeakyReLU(),
        )

        blocks = []
        for _ in range(block_depth - 1):
            blocks.append(nn.Linear(hidden_size, hidden_size))
            blocks.append(nn.LeakyReLU())
        self.blocks = nn.Sequential(
            *blocks, nn.Dropout(dropout), nn.Linear(hidden_size, 1)
        )

    def forward(self, ys, zs):
        h1 = self.label_embed(ys.long())
        h2 = self.z_embed(zs)
        return self.blocks(h1 + h2)

In [37]:
# no MI example
rng = np.random.default_rng()
zs_random = rng.normal(loc=0.0, scale=1.0, size=zs.shape)

# strong MI example
embedder = LabelEncoder(
    label_embed_sizes=label_encoder.label_kwargs["label_embed_sizes"],
    hidden_size=zs.shape[1],
)
zs_strong = embedder(ys).detach().numpy()


results = {}
for name, z_test in [
    ("random", zs_random),
    ("real", zs),
    ("strong", zs_strong),
]:
    logger = TensorBoardLogger("logs", name=name)

    data_loader = DataModule(
        ys=ys,
        zs=z_test,
        val_split=0.1,
        test_split=0.0,
        batch_size=512,
        num_workers=8,
        pin_memory=False,
    )

    net = MinerNet(
        encoder_kwargs=label_encoder.label_kwargs,
        hidden_size=256,
        block_depth=2,
        latent_dim=6,
        dropout=0.3,
    )

    kwargs = {"lr": 1e-3}
    model = MutualInformationEstimator(net=net, **kwargs)
    trainer = Trainer(
        min_epochs=50,
        max_epochs=1000,
        accelerator=device,
        devices=1,
        enable_progress_bar=False,
        logger=logger,
        enable_checkpointing=True,
        callbacks=[
            EarlyStopping(
                monitor="val_loss", patience=50, stopping_threshold=0.0
            ),
            # ModelCheckpoint(
            #     monitor="val_loss", save_top_k=2, save_weights_only=False
            # ),
        ],
    )
    trainer.fit(model, datamodule=data_loader)
    results[name] = trainer.test(ckpt_path="best", datamodule=data_loader)

    print(f"Results for {name}: {results[name]}")

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/torch/utils/data/dataset.py:469: UserWarning: Length of split at index 2 is 0. This might result in an empty dataset.
  warnings.warn(
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (11) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -0.0940566211938858    │
│          test_mi          │    0.0940566211938858     │
└───────────────────────────┴───────────────────────────┘

Results for random: [{'test_loss': -0.0940566211938858, 'test_mi': 0.0940566211938858}]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │   -0.19499719142913818    │
│          test_mi          │    0.19499719142913818    │
└───────────────────────────┴───────────────────────────┘

Results for real: [{'test_loss': -0.19499719142913818, 'test_mi': 0.19499719142913818}]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -3.6700851917266846    │
│          test_mi          │    3.6700851917266846     │
└───────────────────────────┴───────────────────────────┘

Results for strong: [{'test_loss': -3.6700851917266846, 'test_mi': 3.6700851917266846}]


In [34]:
data_loader = DataModule(
    ys=ys,
    zs=zs,
    val_split=0.1,
    test_split=0.1,
    batch_size=512,
    num_workers=8,
    pin_memory=False,
)

results = {}
for lr in [1e-3]:
    for hidden_size in [64, 128, 256]:
        for hidden_depth in [1, 2, 3]:
            for do in [0.0]:
                name = f"lr_{lr}_{do}_{hidden_size}x{hidden_depth}"
                net = MinerNet(
                    encoder_kwargs=label_encoder.label_kwargs,
                    hidden_size=hidden_size,
                    latent_dim=6,
                    block_depth=hidden_depth,
                    dropout=do,
                )

                kwargs = {"lr": lr}

                model = MutualInformationEstimator(net=net, **kwargs).to(device)
                trainer = Trainer(
                    max_epochs=100,
                    accelerator=device,
                    devices=1,
                    enable_progress_bar=False,
                )
                trainer.fit(model, datamodule=data_loader)
                results[name] = trainer.test(datamodule=data_loader)

                print(f"Results for {name}: {results[name]}")

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (10) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/checkpoint_connector.py:149: `.test(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.test(ckpt_path='best')` to use the best model or `.test(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │  4.3964868382317945e-05   │
│          test_mi          │  -4.3964868382317945e-05  │
└───────────────────────────┴───────────────────────────┘

Results for lr_0.001_0.0_64x1: [{'test_loss': 4.3964868382317945e-05, 'test_mi': -4.3964868382317945e-05}]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -0.1962473839521408    │
│          test_mi          │    0.1962473839521408     │
└───────────────────────────┴───────────────────────────┘

Results for lr_0.001_0.0_64x2: [{'test_loss': -0.1962473839521408, 'test_mi': 0.1962473839521408}]


...

Results for lr_0.001_0.0_256x1: [{'test_loss': 0.00010428396490169689, 'test_mi': -0.00010428396490169689}]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -0.5128145217895508    │
│          test_mi          │    0.5128145217895508     │
└───────────────────────────┴───────────────────────────┘

Results for lr_0.001_0.0_256x2: [{'test_loss': -0.5128145217895508, 'test_mi': 0.5128145217895508}]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -0.6875849366188049    │
│          test_mi          │    0.6875849366188049     │
└───────────────────────────┴───────────────────────────┘

Results for lr_0.001_0.0_256x3: [{'test_loss': -0.6875849366188049, 'test_mi': 0.6875849366188049}]
